In [1]:
import pandas as pd

from ambdes import Model, SimConfig, ambsys

In [24]:
def ambsys(csv_path, org_code, month, year):
    """Extract AmbSYS timing metrics for a given organisation, month and year.

    Parameters
    ----------
    csv_path : str
        Path to the AmbSYS CSV file.
    org_code : str
        Organisation code used to filter the dataset.
    month : int
        Month of interest as an integer from 1 to 12.
    year : int
        Year of interest.

    Returns
    -------
    dict
        Nested dictionary containing mean inter-arrival times for C1-C4 calls,
        mean response times for C1-C4 calls, and mean handover time, all
        expressed in minutes.

    """
    # Extract series for given organisation, year and month
    df = pd.read_csv(csv_path)
    month_df = df.loc[
        (df["Org Code"] == org_code)
        & (df["Year"] == int(year))
        & (df["Month"] == int(month))
    ].squeeze()

    if month_df.empty:
        raise ValueError(
            f"No AmbSYS data found for org_code={org_code}, "
            f"year={year}, month={month}."
        )

    return month_df

In [25]:
ambsys_data = ambsys(
    csv_path="data/AmbSYS-to-Mar-2026-UI7FG.csv",
    org_code="RYF",
    month="3",
    year=2026
)
ambsys_data

Year                                                     2026
Month                                                       3
Region                                                    Y58
Org Code                                                  RYF
Org Name    SOUTH WESTERN AMBULANCE SERVICE NHS FOUNDATION...
                                  ...                        
A69                                                         .
A70                                                         .
A71                                                         .
A72                                                         .
A73                                                         .
Name: 1968, Length: 153, dtype: object

0

In [3]:
config = SimConfig(
    ambsys_data=ambsys_data,
    run_length=30,
    log_to_console=True
)
config.mean_iat_min

{1: 5.02646098412341,
 2: 1.042868823735545,
 3: 2.4551754482455177,
 4: 135.6838905775076}

In [4]:
model = Model(run_number=0, config=config)
model.run()

Initialising model for run 0                                                                                       
0.838: Patient 1 (C2) calls                                                                                        
1.236: Patient 2 (C2) calls                                                                                        
1.616: Patient 3 (C2) calls                                                                                        
2.168: Patient 4 (C2) calls                                                                                        
3.757: Patient 5 (C3) calls                                                                                        
4.040: Patient 6 (C3) calls                                                                                        
5.076: Patient 7 (C2) calls                                                                                        
6.932: Patient 8 (C3) calls                                             

In [5]:
model.call_dists

{1: Exponential(mean=5.02646098412341),
 2: Exponential(mean=1.042868823735545),
 3: Exponential(mean=2.4551754482455177),
 4: Exponential(mean=135.6838905775076)}